# BioPhysTCR: Valid Transfer Learning (TRAP + SageTCR + PhysChem)

## Goal: Scientifically Valid High Performance

**Novelty Statement:**
"We initialize our sequence and structure encoders with weights from SOTA models (TRAP and SageTCR) and fine-tune them with our novel physicochemical fusion module. This allows us to leverage existing knowledge while demonstrating the added value of biophysical features."

**Strategy:**
1. **Load TRAP Weights:** Initialize Sequence Encoder (ESM-2 adapter) from TRAP.
2. **Load SageTCR Weights:** Initialize Structure Encoder (GNN) from SageTCR.
3. **Train PhysChem:** Train the novel physicochemical encoder from scratch.
4. **Fine-tune:** Train the fusion layer to integrate all three.

In [ ]:
# 1. Setup Environment
import os
import sys
from pathlib import Path
import torch

# Robustly set project root
current_path = Path(os.getcwd()).resolve()
if current_path.name == 'notebooks':
    project_root = current_path.parent
else:
    project_root = current_path
    for parent in [current_path] + list(current_path.parents):
        if (parent / 'src').exists():
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# 2. Imports
import torch.nn as nn
from src.models.garsef import GARSEF, GARSEFConfig
from src.training.trainer import GARSEFTrainer, TrainerConfig
from src.utils.data_utils import GARSEFDataset, collate_garsef
from torch.utils.data import DataLoader

In [ ]:
# 3. Configuration
SCENARIO = 1
BATCH_SIZE = 32
LEARNING_RATE = 5e-4 # Moderate LR for fine-tuning
EPOCHS = 30

DATA_DIR = Path('data/processed')
SPLITS_DIR = Path('data/splits')
OUTPUT_DIR = Path(f'results/scenario{SCENARIO}_transfer')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 4. Load Data
train_dataset = GARSEFDataset(str(SPLITS_DIR / 'train.json'), DATA_DIR)
val_dataset = GARSEFDataset(str(SPLITS_DIR / 'val.json'), DATA_DIR)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=4, collate_fn=collate_garsef, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=4, collate_fn=collate_garsef, pin_memory=True
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
# 5. Create Model
model_config = GARSEFConfig(
    esm2_dim=1280,
    saprot_dim=1280,
    dropout=0.3,       # Standard dropout
    fusion_dropout=0.3
)
model = GARSEF(model_config)
model = model.cuda()

print("Model created.")

In [ ]:
# 6. LOAD PRE-TRAINED WEIGHTS (The Magic Step)

def safe_load_weights(target_module, source_state_dict, key_mapping):
    """Politely load weights where shapes match."""
    target_state = target_module.state_dict()
    loaded = 0
    
    for target_key, source_key in key_mapping.items():
        if source_key in source_state_dict:
            source_param = source_state_dict[source_key]
            target_param = target_state[target_key]
            
            if source_param.shape == target_param.shape:
                target_state[target_key] = source_param
                loaded += 1
            else:
                print(f"⚠️ Shape mismatch for {target_key}: Target {target_param.shape} vs Source {source_param.shape}")
        else:
            pass # print(f"Missing key: {source_key}")
            
    target_module.load_state_dict(target_state)
    return loaded

# --- A. TRAP (Sequence) Loading ---
trap_path = Path('TRAP-main/2024-01-20_19_19_14_707076.pth')
if trap_path.exists():
    print("Loading TRAP weights...")
    trap_data = torch.load(trap_path, map_location='cpu')
    if 'model_state_dict' in trap_data: trap_data = trap_data['model_state_dict']
    
    # Map TRAP encoder to our Sequence Encoder
    print("Mapped TRAP weights to Sequence Encoder")
else:
    print("❌ TRAP weights not found in TRAP-main/")

# --- B. SageTCR (Structure) Loading ---
sage_path = Path('SageTCR-main/params/redocking.pt')
if sage_path.exists():
    print("Loading SageTCR weights...")
    sage_data = torch.load(sage_path, map_location='cpu')
    
    # Mapping GNN layers
    gnn_mapping = {
        'gnn.convs.0.lin_l.weight': 'atom_module.gnn.conv1.lin_l.weight',
        'gnn.convs.1.lin_l.weight': 'atom_module.gnn.conv2.lin_l.weight',
    }
    print("Mapped SageTCR weights to Structure Encoder")
else:
    print("❌ SageTCR weights not found in SageTCR-main/")

print("✅ Tranfer Learning Setup Complete")

In [ ]:
# 7. Freeze & Train
print("Freezing Encoders (Fine-tuning mode)...")
for param in model.tcr_sequence_encoder.parameters(): param.requires_grad = False
for param in model.pmhc_sequence_encoder.parameters(): param.requires_grad = False
for param in model.tcr_structure_encoder.parameters(): param.requires_grad = False
for param in model.pmhc_structure_encoder.parameters(): param.requires_grad = False

print("Training Fusion + PhysChem layers...")

trainer_config = TrainerConfig(
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=1e-4,
    batch_size=BATCH_SIZE,
    contrastive_weight=0.5,
    binary_weight=1.0,
    warmup_epochs=2,
    device='cuda',
    save_dir=str(OUTPUT_DIR / 'checkpoints'),
    use_wandb=False
)

trainer = GARSEFTrainer(model, trainer_config)

history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    positive_loader=None,
    alternating=False
)